# Đánh giá VLM đọc tiếng Việt: Vintern, PaddleOCR-VL, Qwen-VL

Cho một số VLM (model đọc được cả ảnh và chữ) phổ biến đọc golden set do `src/data-golden-set-generator` sinh ra, chấm với `ground_truth.json` theo cách của `ocr-eval.ipynb` để so với OCR chuyên dụng. Có hai chế độ, ứng với hai hướng VLM trong AIP-001:

- **Cả trang:** VLM đọc cả trang giấy tờ (hướng "VLM đọc cả trang").
- **Recognition-only:** cắt đúng bbox ground truth rồi cho VLM đọc. Đây là phần cascade AIP-035: OCR thường đọc trước, vùng có điểm thấp mới đưa VLM đọc lại.

| Model | Cỡ | Ghi chú |
|---|---|---|
| `5CD-AI/Vintern-1B-v3_5` | 1B | InternVL2.5 fine-tune cho tiếng Việt, remote code, gọi qua `model.chat` |
| `PaddlePaddle/PaddleOCR-VL` | 0.9B | VLM chuyên OCR, 109 ngôn ngữ có tiếng Việt, prompt cố định `OCR:` |
| `Qwen/Qwen3-VL-2B/4B/8B-Instruct` | 2–8B | Qwen3-VL, OCR 32 ngôn ngữ có tiếng Việt |
| `Qwen/Qwen2.5-VL-3B/7B-Instruct` | 3–7B | Hay được dùng làm model nền cho các model OCR khác |
| `google/gemma-3-4b-it` | 4B | Đa ngôn ngữ. Gated: phải chấp nhận license trên Hugging Face |

Khác với `ocr-eval.ipynb`:

- VLM không trả bbox. Ở chế độ cả trang, mỗi trường được tìm theo **đoạn con khớp nhất trong cả trang**, không giới hạn theo vị trí. Cách này dễ hơn cách khớp của `ocr-eval.ipynb` (trường ngắn như `1`, `Nam` dễ khớp nhầm chỗ khác), nên **không so số cả trang với OCR**. Số rec-only thì so trực tiếp được (mục 6.7).
- `score` = trung bình nhân xác suất các token sinh ra (greedy). Vintern gọi qua `model.chat` nên không có score, không tính ECE.
- VLM có lỗi mà OCR không có: **lặp vô hạn** (sinh tới `max_new_tokens`, cột "Trang bị cắt") và **bịa thêm chữ** (cột "Đọc thừa": kết quả dài hơn 1,5 lần ground truth).

Trường `checkbox` bị loại. Dữ liệu tổng hợp nên metric đẹp hơn hồ sơ thật: dùng để so model, không dùng làm con số cuối.

## 0. Cài đặt

Local: dùng `jupyters/.venv` (xem `README.md`), cài thêm `transformers`, `accelerate`, `timm`. Colab: chọn runtime GPU, upload và giải nén dữ liệu vào `/content/data` rồi chạy cell dưới.

In [ ]:
import os, sys
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ  # Kaggle: bật Internet trong Settings để pip cài được
if "google.colab" in sys.modules or ON_KAGGLE:
    # giữ nguyên torch có sẵn: pip đổi torch khi kernel đã nạp torch -> lỗi TORCH_LIBRARY triton, phải restart
    import importlib.metadata
    open("/tmp/torch-pin.txt", "w").write(f"torch=={importlib.metadata.version('torch')}\n")
    %pip install -q -c /tmp/torch-pin.txt "transformers>=4.57,<5" accelerate timm einops rapidfuzz
    # %pip lỗi mạng chỉ in WARNING rồi chạy tiếp: dừng ở đây cho rõ nguyên nhân
    import importlib.util
    if importlib.util.find_spec("rapidfuzz") is None:
        raise RuntimeError("pip install thất bại. Kaggle: bật Settings > Internet (cần xác minh số điện thoại), rồi Restart & Run All")
    # transformers đã import sẵn bản khác (Colab cài sẵn 5.x): phải restart kernel mới nạp bản vừa cài
    if "transformers" in sys.modules and sys.modules["transformers"].__version__ != importlib.metadata.version("transformers"):
        print("Đã đổi bản transformers, runtime sẽ khởi động lại. Chạy lại notebook từ đầu (Run all).")
        os.kill(os.getpid(), 9)

## 1. Cấu hình

`MODELS` có dạng `<engine>/<repo Hugging Face>`. Bỏ dòng nào thì không chạy model đó.

| Engine | Chạy bằng | Dùng cho |
|---|---|---|
| `hf` | `transformers`, `AutoModelForImageTextToText` và chat template | Qwen-VL, Gemma 3, PaddleOCR-VL |
| `vintern` | `transformers` remote code, `model.chat`, chia ô 448px như model card | Vintern (InternVL) |
| `vllm` | server tương thích OpenAI ở `VLLM_URL` (`vllm serve <repo>`), gửi nhiều request song song | model lớn, chạy nhanh trên GPU |

Trên CPU mỗi trang mất vài phút với model 2–4B (phần lớn là sinh vài trăm token), nên mặc định `MAX_IMAGES = 6`. GPU thì chạy hết. Ảnh trang được thu nhỏ để cạnh dài ≤ `PAGE_MAX_SIDE`, vì số token ảnh tăng theo số pixel.

In [ ]:
import os
import base64, gc, io, json, re, time, unicodedata, urllib.request
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from functools import cache
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from rapidfuzz.distance import Levenshtein

KAGGLE_DATA = Path("/kaggle/input/datasets/balebomm/dataset/generated")  # dataset Kaggle, chỉ đọc
DATA_DIR = KAGGLE_DATA if KAGGLE_DATA.is_dir() else Path("data")  # thư mục generated/ của generator (Colab: /content/data)
OUTPUT_DIR = Path("outputs")             # cache: outputs/<engine>/<repo>/, cùng chỗ với ocr-eval.ipynb
# <doc-id>_p<N>.png (sạch), <doc-id>_p<N>_<level>.jpg (nhiễu). Ảnh *_bbox.jpg không khớp
IMAGE_PATTERN = re.compile(r"^(?P<doc_id>.+)_p(?P<page>\d+)(?:\.png|_(?P<level>light|medium|heavy)\.jpg)$")
LEVELS = ["clean", "light", "medium", "heavy"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = DEVICE == "cuda"
# bf16 cho model VLM ổn định hơn fp16 (Qwen fp16 dễ tràn số). T4 (Colab free) không có bf16 thật, dùng fp16
DTYPE = torch.float16 if GPU and torch.cuda.get_device_capability()[0] < 8 else torch.bfloat16
MAX_IMAGES = None if GPU else 6          # None = tất cả

MODELS = [
    "vintern/5CD-AI/Vintern-1B-v3_5",
    "hf/PaddlePaddle/PaddleOCR-VL",
    "hf/Qwen/Qwen3-VL-2B-Instruct",
    "hf/Qwen/Qwen2.5-VL-3B-Instruct",
    # Cần GPU hoặc RAM lớn (bf16 khoảng 2GB mỗi tỷ tham số)
    # "hf/Qwen/Qwen3-VL-4B-Instruct",
    # "hf/Qwen/Qwen3-VL-8B-Instruct",
    # "hf/Qwen/Qwen2.5-VL-7B-Instruct",
    # "hf/google/gemma-3-4b-it",          # gated: chấp nhận license, đặt biến môi trường HF_TOKEN
    # "vllm/Qwen/Qwen2.5-VL-7B-Instruct", # chạy trước: vllm serve Qwen/Qwen2.5-VL-7B-Instruct
]

PAGE_PROMPT = ("Đọc toàn bộ chữ trong ảnh giấy tờ này, từ trên xuống dưới, từ trái sang phải. "
               "Giữ nguyên chính tả và dấu tiếng Việt, không sửa, không dịch, không giải thích. "
               "Mỗi dòng chữ trong ảnh là một dòng kết quả.")
LINE_PROMPT = "Đọc chữ tiếng Việt trong ảnh. Chỉ trả về đúng chữ có trong ảnh, giữ nguyên dấu, không thêm gì khác."
PROMPTS = {"PaddlePaddle/PaddleOCR-VL": ("OCR:", "OCR:")}  # (trang, dòng). Model chỉ được train với prompt cố định

# Tham số from_pretrained riêng từng model. PaddleOCR-VL dùng remote code; nếu remote code lỗi với bản
# transformers đang cài, thử bỏ trust_remote_code (các bản transformers mới có sẵn PaddleOCR-VL)
HF_OPTS = {"PaddlePaddle/PaddleOCR-VL": dict(trust_remote_code=True)}

PAGE_MAX_SIDE = 1600 if GPU else 1280    # px, cạnh dài của trang đưa vào model
PAGE_MAX_TOKENS = 2048                   # trang đầy chữ khoảng 800–1500 token
LINE_MIN_SIDE, LINE_MAX_SIDE = 64, 1024  # vùng cắt: phóng để cao ≥ 64px (Qwen chia patch 28px), cạnh dài ≤ 1024px
LINE_MAX_TOKENS = 96
PAD = 6                                  # px nới quanh bbox khi cắt, giống ocr-eval.ipynb
REC_BATCH = 8 if GPU else 4              # số vùng cắt sinh cùng lúc (engine hf)
REC_CHUNK = 64                           # ghi cache rec-only sau mỗi chừng này trường, dừng giữa chừng không mất kết quả

VINTERN_PAGE_TILES, VINTERN_LINE_TILES = (12 if GPU else 6), 4   # số ô 448px tối đa
VINTERN_REPETITION_PENALTY = 1.1         # model card dùng 2.5 + beam 3, hợp tóm tắt hơn đọc nguyên văn

VLLM_URL = os.environ.get("VLLM_URL", "http://localhost:8000/v1")
VLLM_WORKERS = 8                         # số request song song

ENGINE_NAMES = list(dict.fromkeys(m.split("/")[0] for m in MODELS))
pd.set_option("display.max_colwidth", 80)
print(f"{DEVICE}, {DTYPE}, {len(MODELS)} model")

In [ ]:
# Biểu đồ: mỗi model một màu theo thứ tự cố định, thêm marker để không phụ thuộc màu. OCR chuyên dụng (mục 6.7) màu xám
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
MARKERS = ["o", "s", "D", "^", "v", "P", "X", "*"]
COLOR = {m: SERIES[i % len(SERIES)] for i, m in enumerate(MODELS)}
MARKER = {m: MARKERS[i % len(MARKERS)] for i, m in enumerate(MODELS)}
LABEL = {m: m.split("/", 1)[1].split("/")[-1] for m in MODELS}   # tên ngắn trên biểu đồ
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
INK, MUTED, GRID, SURFACE, BASELINE = "#1f2328", "#59636e", "#d8dee4", "#fcfcfb", "#9aa3ad"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
                     "axes.titlesize": 11, "axes.labelcolor": MUTED, "text.color": INK})
pct = plt.FuncFormatter(lambda v, _: f"{v * 100:.4g}%")

def style_axes(ax, grid="y", percent_axis="y"):
    ax.grid(axis=grid, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED, length=0)
    if percent_axis:
        getattr(ax, f"{percent_axis}axis").set_major_formatter(pct)

def legend(fig, models=MODELS):
    handles = [plt.Line2D([], [], color=COLOR[m], linewidth=2, marker=MARKER[m], markersize=7,
                          markeredgecolor="white") for m in models]
    fig.legend(handles, [LABEL[m] for m in models], frameon=False, ncol=min(len(models), 3),
               loc="lower center", bbox_to_anchor=(0.5, -0.1))

## 2. Dữ liệu và ground truth

Giống `ocr-eval.ipynb`: quét đệ quy `DATA_DIR` lấy ảnh khớp `IMAGE_PATTERN`, tra ground truth từ mọi `ground_truth.json`. Mỗi dòng của `items` là một trường trên một ảnh.

- **Local:** `ln -s ../src/data-golden-set-generator/generated data`
- **Colab:** zip nội dung bên trong `generated/` thành `generated.zip`, upload vào `/content`. Cell dưới tự giải nén vào `data/` nếu chưa có
- **Kaggle:** add dataset `balebomm/dataset`, notebook tự dùng `/kaggle/input/datasets/balebomm/dataset/generated`. Kết quả ghi vào `outputs/` trong `/kaggle/working`

In [ ]:
DATA_ZIP = Path("generated.zip")          # nội dung bên trong generated/, upload cạnh notebook (Colab: /content)
if not DATA_DIR.is_dir() and DATA_ZIP.exists():
    import zipfile
    print(f"Giải nén {DATA_ZIP} -> {DATA_DIR}/")
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(DATA_DIR)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Không thấy {DATA_DIR.resolve()} và {DATA_ZIP.resolve()}. Xem hướng dẫn đặt dữ liệu ở trên.")

images = sorted(p.as_posix() for p in DATA_DIR.rglob("*") if IMAGE_PATTERN.match(p.name))[:MAX_IMAGES]

def index_ground_truth(data_dir: Path) -> dict:
    """Đường dẫn ảnh -> (thông tin giấy tờ, level, page, bbox theo tên trường)."""
    index = {}
    for gt_path in data_dir.rglob("ground_truth.json"):
        gt = json.loads(gt_path.read_text(encoding="utf-8"))
        for doc in gt["documents"]:
            clean_boxes = {f["name"]: f.get("bbox", []) for f in doc["fields"]}
            pages = [("clean", i + 1, p, clean_boxes) for i, p in enumerate(doc["files"]["pages"])]
            pages += [(n["level"], n["page"], n["path"], n["bboxes"]) for n in doc["files"].get("noisy", [])]
            for level, page, rel, boxes in pages:
                index[(gt_path.parent / rel).as_posix()] = (gt["dossier_id"], doc, level, page, boxes)
    return index

def group_lines(boxes):
    """Gom các bbox nằm ở dòng kế tiếp nhau (trường dài bị xuống dòng) thành một lần xuất hiện.
    Bbox cách xa nhau (trường in ở nhiều chỗ, ví dụ tên ở đầu trang và chỗ ký) vẫn tách riêng."""
    groups = []
    for b in sorted(boxes, key=lambda b: (b["y"], b["x"])):
        if groups:
            p = groups[-1][-1]
            next_line = b["y"] > p["y"] + p["h"] * 0.5 and b["y"] - (p["y"] + p["h"]) < p["h"] * 0.8
            if next_line:
                groups[-1].append(b)
                continue
        groups.append([b])
    return groups

def build_items(images, gt_index) -> pd.DataFrame:
    rows = []
    for im in images:
        if im not in gt_index:
            continue
        dossier, doc, level, page, boxes = gt_index[im]
        for f in doc["fields"]:
            if f.get("fill") == "checkbox" or not f["text"].strip():
                continue
            page_boxes = [bb for bb in boxes.get(f["name"], []) if bb.get("page", 1) == page]
            for k, group in enumerate(group_lines(page_boxes)):
                x0, y0 = min(b["x"] for b in group), min(b["y"] for b in group)
                x1, y1 = max(b["x"] + b["w"] for b in group), max(b["y"] + b["h"] for b in group)
                rows.append(dict(
                    dossier=dossier, doc_id=doc["doc_id"], doc_type=doc["doc_type"], level=level, image=im,
                    field=f["name"], field_type=f["type"], fill=f.get("fill", "printed"),
                    occurrence=k, truth=f["text"], n_lines=len(group),
                    boxes=[(b["x"], b["y"], b["w"], b["h"]) for b in group],
                    x=x0, y=y0, w=x1 - x0, h=y1 - y0,     # khung bao, dùng để cắt ảnh xem
                ))
    return pd.DataFrame(rows)

gt_index = index_ground_truth(DATA_DIR)
items = build_items(images, gt_index)
missing = [im for im in images if im not in gt_index]
if missing:
    print(f"Bỏ qua {len(missing)} ảnh không có trong ground_truth.json nào, ví dụ {missing[0]}")
    images = [im for im in images if im in gt_index]
if items.empty:
    raise RuntimeError(f"Không tìm thấy ảnh generator trong {DATA_DIR.resolve()}")
items["level"] = pd.Categorical(items["level"], LEVELS, ordered=True)
REC_KEYS = [f"{Path(r.image).relative_to(DATA_DIR).as_posix()}|{r.field}|{r.occurrence}" for r in items.itertuples()]
print(f"{len(images)} ảnh, {items.dossier.nunique()} hồ sơ, {len(items)} trường cần đọc, "
      f"{sum(len(b) for b in items.boxes)} vùng cắt rec-only mỗi model")
items.groupby(["doc_type", "fill"]).field.unique().to_frame("fields")

## 3. Engine

Mỗi engine có 2 hàm tạo, nhận repo và trả về hàm đọc. Model chỉ tải khi đọc ảnh đầu tiên.

- `page(repo)` → `f(image_path)`: đọc cả trang.
- `line(repo)` → `f([ảnh PIL])`: đọc một lô vùng cắt.

Cả hai trả `(text, score, truncated, n_tokens)`: `truncated` là sinh tới giới hạn token mà chưa kết thúc (thường do lặp).

In [ ]:
def fit(img: Image.Image, max_side: int, min_side: int = 0) -> Image.Image:
    """Thu nhỏ để cạnh dài ≤ max_side (giới hạn số token ảnh), phóng to để cạnh ngắn ≥ min_side."""
    w, h = img.size
    s = min(1, max_side / max(w, h))
    if min_side and min(w, h) * s < min_side:
        s = min(min_side / min(w, h), max_side / max(w, h) * 2)   # dòng rất dài: chấp nhận vượt max_side tới 2 lần
    return img if s == 1 else img.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BICUBIC)

def prompts(repo):
    return PROMPTS.get(repo, (PAGE_PROMPT, LINE_PROMPT))

def batched(fn, crops, size):
    return [r for i in range(0, len(crops), size) for r in fn(crops[i:i + size])]

# --- hf: transformers, chat template chung
def to_device(inputs):
    return {k: (v.to(DEVICE, DTYPE) if v.is_floating_point() else v.to(DEVICE)) if torch.is_tensor(v) else v
            for k, v in inputs.items()}

@cache
def hf_model(repo):
    from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor
    opts = HF_OPTS.get(repo, {})
    processor = AutoProcessor.from_pretrained(repo, **opts)
    processor.tokenizer.padding_side = "left"          # sinh theo lô: pad bên trái
    try:
        model = AutoModelForImageTextToText.from_pretrained(repo, dtype=DTYPE, **opts)
    except ValueError:                                  # remote code chỉ đăng ký AutoModelForCausalLM
        model = AutoModelForCausalLM.from_pretrained(repo, dtype=DTYPE, **opts)
    return processor, model.to(DEVICE).eval()

def hf_generate(repo, imgs, prompt, max_new_tokens):
    """Sinh greedy cho một lô ảnh cùng prompt. score = trung bình nhân xác suất các token tới token kết thúc."""
    processor, model = hf_model(repo)
    convs = [[{"role": "user", "content": [{"type": "image", "image": im}, {"type": "text", "text": prompt}]}]
             for im in imgs]
    inputs = to_device(processor.apply_chat_template(convs, add_generation_prompt=True, tokenize=True,
                                                     return_dict=True, return_tensors="pt", padding=True))
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
                             return_dict_in_generate=True, output_scores=True)
    gen = out.sequences[:, inputs["input_ids"].shape[1]:]
    logp = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True).float()
    eos = model.generation_config.eos_token_id
    is_end = torch.isin(gen, torch.tensor([eos] if isinstance(eos, int) else list(eos), device=gen.device))
    valid = (is_end.cumsum(1) - is_end.long()) == 0     # bỏ token pad sau token kết thúc đầu tiên
    texts = processor.batch_decode(gen, skip_special_tokens=True)
    return [(t, float(torch.exp(lp[v].mean())) if v.any() else 0.0, not bool(e.any()), int(v.sum()))
            for t, lp, v, e in zip(texts, logp, valid, is_end)]

def hf_page(repo):
    prompt = prompts(repo)[0]
    return lambda path: hf_generate(repo, [fit(Image.open(path).convert("RGB"), PAGE_MAX_SIDE)], prompt, PAGE_MAX_TOKENS)[0]

def hf_line(repo):
    prompt, batch = prompts(repo)[1], [REC_BATCH]
    def run(crops):
        try:
            return batched(lambda c: hf_generate(repo, c, prompt, LINE_MAX_TOKENS), crops, batch[0])
        except Exception as e:                          # processor remote code không nhận lô: đọc từng ảnh
            if batch[0] == 1:
                raise
            print(f"{repo}: sinh theo lô lỗi ({type(e).__name__}: {e}), chuyển sang từng ảnh")
            batch[0] = 1
            return run(crops)
    return run

# --- vintern: InternVL remote code, chia ảnh thành ô 448px (dynamic_preprocess trong model card)
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def vintern_tiles(img, max_tiles, size=448):
    import torchvision.transforms as T
    w, h = img.size
    ratios = sorted({(c, r) for c in range(1, max_tiles + 1) for r in range(1, max_tiles + 1) if c * r <= max_tiles},
                    key=lambda cr: cr[0] * cr[1])
    best, best_diff = (1, 1), float("inf")
    for c, r in ratios:                                 # tỷ lệ gần nhất; hòa thì lấy nhiều ô hơn nếu ảnh đủ lớn
        diff = abs(w / h - c / r)
        if diff < best_diff or (diff == best_diff and w * h > 0.5 * size * size * c * r):
            best, best_diff = (c, r), diff
    cols, rows = best
    resized = img.resize((size * cols, size * rows))
    tiles = [resized.crop((c * size, r * size, (c + 1) * size, (r + 1) * size)) for r in range(rows) for c in range(cols)]
    if len(tiles) > 1:
        tiles.append(img.resize((size, size)))         # ảnh thu nhỏ toàn cảnh
    tf = T.Compose([T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC), T.ToTensor(),
                    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
    return torch.stack([tf(t.convert("RGB")) for t in tiles])

@cache
def vintern_model(repo):
    import transformers
    from transformers import AutoModel, AutoTokenizer
    if int(transformers.__version__.split(".")[0]) >= 5:
        raise RuntimeError(f"Remote code Vintern cần transformers < 5, đang chạy {transformers.__version__}. "
                           "Cài transformers>=4.57,<5 rồi restart runtime")
    model = AutoModel.from_pretrained(repo, dtype=DTYPE, trust_remote_code=True, use_flash_attn=False, low_cpu_mem_usage=True)
    tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True, use_fast=False)
    return tok, model.to(DEVICE).eval()

def vintern_generate(repo, img, prompt, max_new_tokens, max_tiles):
    tok, model = vintern_model(repo)
    cfg = dict(max_new_tokens=max_new_tokens, do_sample=False, num_beams=1, repetition_penalty=VINTERN_REPETITION_PENALTY)
    with torch.inference_mode():
        text = model.chat(tok, vintern_tiles(img, max_tiles).to(DEVICE, DTYPE), "<image>\n" + prompt, cfg)
    n = len(tok(text).input_ids)                         # model.chat chỉ trả chữ: đếm lại token, không có score
    return text, float("nan"), n >= max_new_tokens - 2, n

def vintern_page(repo):
    prompt = prompts(repo)[0]
    return lambda path: vintern_generate(repo, Image.open(path).convert("RGB"), prompt, PAGE_MAX_TOKENS, VINTERN_PAGE_TILES)

def vintern_line(repo):
    prompt = prompts(repo)[1]
    return lambda crops: [vintern_generate(repo, c, prompt, LINE_MAX_TOKENS, VINTERN_LINE_TILES) for c in crops]

# --- vllm: server tương thích OpenAI (vLLM, SGLang, LM Studio...). Không cần torch ở phía notebook
def vllm_generate(repo, img, prompt, max_new_tokens):
    buf = io.BytesIO()
    img.save(buf, "PNG")
    body = dict(model=repo, temperature=0, max_tokens=max_new_tokens, logprobs=True, messages=[{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}},
        {"type": "text", "text": prompt}]}])
    req = urllib.request.Request(VLLM_URL.rstrip("/") + "/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json",
                                          "Authorization": f"Bearer {os.environ.get('VLLM_API_KEY', 'EMPTY')}"})
    with urllib.request.urlopen(req, timeout=900) as r:
        resp = json.load(r)
    ch = resp["choices"][0]
    lps = [t["logprob"] for t in ((ch.get("logprobs") or {}).get("content") or [])]
    return (ch["message"]["content"] or "", float(np.exp(np.mean(lps))) if lps else float("nan"),
            ch["finish_reason"] == "length", resp["usage"]["completion_tokens"])

def vllm_page(repo):
    prompt = prompts(repo)[0]
    return lambda path: vllm_generate(repo, fit(Image.open(path).convert("RGB"), PAGE_MAX_SIDE), prompt, PAGE_MAX_TOKENS)

def vllm_line(repo):
    prompt = prompts(repo)[1]
    def run(crops):
        with ThreadPoolExecutor(VLLM_WORKERS) as ex:
            return list(ex.map(lambda c: vllm_generate(repo, c, prompt, LINE_MAX_TOKENS), crops))
    return run

ENGINES = {"hf": (hf_page, hf_line), "vintern": (vintern_page, vintern_line), "vllm": (vllm_page, vllm_line)}

def free_models():
    for f in (hf_model, vintern_model):
        f.cache_clear()
    gc.collect()
    if GPU:
        torch.cuda.empty_cache()

## 4. Chạy model

Mỗi model chạy xong cả trang lẫn rec-only rồi mới giải phóng, để RAM/VRAM chỉ giữ một model. Kết quả cache trong `outputs/<engine>/<repo>/`: mỗi trang một JSON, rec-only gộp ở `_rec_only.json`. Xóa thư mục để chạy lại. Thêm model mới chỉ phải chạy model đó.

Kết quả thô được lưu nguyên. Bỏ định dạng markdown (rào ```` ``` ````, bảng `|`, `**đậm**`, thẻ HTML) làm ở bước chấm.

In [ ]:
def cache_path(model, image):
    return OUTPUT_DIR / model / Path(image).relative_to(DATA_DIR).with_suffix(".json")

def rec_cache_path(model):
    return OUTPUT_DIR / model / "_rec_only.json"

def crop_box(img, x, y, w, h):
    H, W = img.shape[:2]
    return img[max(int(y) - PAD, 0):min(int(y + h) + PAD, H), max(int(x) - PAD, 0):min(int(x + w) + PAD, W)]

def to_pil(img_bgr):
    return Image.fromarray(cv2.cvtColor(np.ascontiguousarray(img_bgr), cv2.COLOR_BGR2RGB))

def run_pages(model, read_page):
    todo = [im for im in images if not cache_path(model, im).exists()]
    print(f"{model}: cả trang {len(images) - len(todo)} ảnh đã cache, chạy {len(todo)} ảnh")
    for i, im in enumerate(todo, 1):
        t0 = time.perf_counter()
        text, score, truncated, n_tokens = read_page(im)
        out = dict(text=text, score=score, truncated=truncated, tokens=n_tokens, seconds=time.perf_counter() - t0)
        cp = cache_path(model, im)
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")
        print(f"  {i}/{len(todo)} {out['seconds']:.0f}s, {n_tokens} token{', BỊ CẮT' if truncated else ''}")

def run_rec(model, read_lines):
    cp = rec_cache_path(model)
    cache_ = json.loads(cp.read_text(encoding="utf-8")) if cp.exists() else {}
    todo = [(k, r) for k, r in zip(REC_KEYS, items.itertuples()) if k not in cache_]
    print(f"{model}: rec-only {len(items) - len(todo)} trường đã cache, chạy {len(todo)} trường")
    imgs = {}
    for i in range(0, len(todo), REC_CHUNK):
        chunk = todo[i:i + REC_CHUNK]
        crops = [fit(to_pil(crop_box(imgs.setdefault(r.image, cv2.imread(r.image)), *box)), LINE_MAX_SIDE, LINE_MIN_SIDE)
                 for _, r in chunk for box in r.boxes]
        t0 = time.perf_counter()
        res = iter(read_lines(crops))
        seconds = (time.perf_counter() - t0) / len(chunk)
        for k, r in chunk:                             # trường nhiều dòng: đọc từng dòng rồi nối, giống ocr-eval.ipynb
            parts = [next(res) for _ in r.boxes]
            cache_[k] = dict(text=" ".join(p[0].strip() for p in parts), score=float(np.mean([p[1] for p in parts])),
                             truncated=any(p[2] for p in parts), seconds=seconds)
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(cache_, ensure_ascii=False), encoding="utf-8")
        print(f"  {min(i + REC_CHUNK, len(todo))}/{len(todo)} ({seconds:.1f}s/trường)")

for m in MODELS:
    engine, repo = m.split("/", 1)
    page, line = ENGINES[engine]
    run_pages(m, page(repo))
    run_rec(m, line(repo))
    free_models()

## 5. Chấm điểm

- **Cả trang:** bỏ định dạng markdown, gộp cả trang thành một chuỗi, mỗi trường tìm đoạn con khớp nhất (khoảng cách semi-global) trong cả chuỗi. `hyp` là đoạn khớp đó.
- **Rec-only:** so toàn chuỗi như `ocr-eval.ipynb`. `score` là score của model trên vùng cắt.

Ghi `outputs/scores_vlm_e2e.csv`, `outputs/scores_vlm_rec.csv` (tên khác `scores_e2e.csv` của `ocr-eval.ipynb` để `dataset-profile.ipynb` không đọc nhầm).

In [ ]:
FENCE = re.compile(r"^```[\w-]*\s*|\s*```$")

def clean_output(text: str) -> str:
    """Bỏ định dạng VLM hay thêm: rào code, thẻ HTML, bảng markdown, **đậm**, # tiêu đề, gạch đầu dòng, ngoặc kép bao ngoài."""
    text = re.sub(r"<[^>]+>", " ", FENCE.sub("", text.strip()))
    lines = []
    for ln in text.splitlines():
        if re.fullmatch(r"[\s|:\-]*", ln):              # dòng trống, dòng kẻ bảng |---|---|
            continue
        ln = re.sub(r"^\s*(#+\s|[-*•]\s|>\s?)", "", ln).replace("**", "")
        lines.append(" ".join(c.strip() for c in ln.strip().strip("|").split("|")))
    return "\n".join(lines).strip().strip('"“”')

def norm(s: str) -> str:
    return " ".join(unicodedata.normalize("NFC", s).split())

def strip_marks(s: str) -> str:
    s = unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").casefold()

def semiglobal(gt, hyp):
    """(số phép sửa ít nhất để gt thành một đoạn con liên tiếp của hyp, vị trí kết thúc đoạn đó trong hyp).
    Nhận chuỗi ký tự hoặc danh sách từ. Mỗi hàng DP tính bằng numpy: phép chèn là cummin của (tmp - j) + j."""
    if not gt:
        return 0, 0
    vocab = {t: i for i, t in enumerate(dict.fromkeys([*gt, *hyp]))}
    h = np.array([vocab[t] for t in hyp], dtype=np.int64)
    j = np.arange(len(hyp) + 1)
    prev = np.zeros(len(hyp) + 1, dtype=np.int64)
    for i, g in enumerate(gt, 1):
        tmp = np.empty_like(prev)
        tmp[0] = i
        tmp[1:] = np.minimum(prev[1:] + 1, prev[:-1] + (h != vocab[g]))
        prev = np.minimum.accumulate(tmp - j) + j
    end = len(prev) - 1 - int(prev[::-1].argmin())    # hòa thì lấy điểm cuối xa nhất: đoạn khớp dài hơn
    return int(prev[end]), end

def best_match(gt: str, text: str) -> str:
    """Đoạn của text khớp gt nhất: tìm điểm cuối, rồi tìm điểm đầu bằng cách chạy ngược."""
    _, end = semiglobal(gt, text)
    _, back = semiglobal(gt[::-1], text[:end][::-1])
    return text[end - back:end]

def score_e2e(model):
    out = []
    for im, g in items.groupby("image", sort=False):
        page = json.loads(cache_path(model, im).read_text(encoding="utf-8"))
        text = norm(clean_output(page["text"]))
        text_nomark, words = strip_marks(text), text.split()
        for idx, r in g.iterrows():
            gt = norm(r.truth)
            out.append(dict(
                idx=idx, hyp=best_match(gt, text), score=page["score"], truncated=page["truncated"],
                cer=min(semiglobal(gt, text)[0] / len(gt), 1),
                cer_nomark=min(semiglobal(strip_marks(gt), text_nomark)[0] / len(gt), 1),
                wer=min(semiglobal(gt.split(), words)[0] / len(gt.split()), 1),
            ))
    return items.assign(model=model, **pd.DataFrame(out).set_index("idx"))

def score_rec(model):
    cache_ = json.loads(rec_cache_path(model).read_text(encoding="utf-8"))
    hyp, gt = [norm(clean_output(cache_[k]["text"])) for k in REC_KEYS], [norm(g) for g in items.truth]
    lev = lambda a, b: min(Levenshtein.distance(a, b) / len(a), 1)
    return items.assign(
        model=model, hyp=hyp, score=[cache_[k]["score"] for k in REC_KEYS],
        truncated=[cache_[k]["truncated"] for k in REC_KEYS], seconds=[cache_[k]["seconds"] for k in REC_KEYS],
        cer=[lev(g, h) for g, h in zip(gt, hyp)],
        cer_nomark=[lev(strip_marks(g), strip_marks(h)) for g, h in zip(gt, hyp)],
        wer=[lev(g.split(), h.split()) for g, h in zip(gt, hyp)],
        over=[len(h) > 1.5 * len(g) + 2 for g, h in zip(gt, hyp)],   # đọc thừa: bịa thêm chữ, lặp, lẫn chữ bên cạnh
    )

e2e = pd.concat([score_e2e(m) for m in MODELS], ignore_index=True)
rec = pd.concat([score_rec(m) for m in MODELS], ignore_index=True)
for df in (e2e, rec):
    df["exact"] = df.cer == 0
pages = pd.DataFrame([dict(model=m, image=im, **json.loads(cache_path(m, im).read_text(encoding="utf-8")))
                      for m in MODELS for im in images])

for name, df in [("e2e", e2e), ("rec", rec)]:
    df.drop(columns="boxes").to_csv(OUTPUT_DIR / f"scores_vlm_{name}.csv", index=False)

## 6. Kết quả

### 6.1. Bảng tổng hợp

Đậm là giá trị tốt nhất mỗi cột. Thời gian chỉ so được giữa các model chạy cùng thiết bị. ECE trống là model không có score (Vintern).

In [ ]:
MAIN = [("CER", False), ("WER", False), ("Field Accuracy", True), ("Document Accuracy", True)]

def main_metrics(df, by=()):
    keys = ["model", *by]
    fields = df.groupby(keys, observed=True).agg(**{"CER": ("cer", "mean"), "WER": ("wer", "mean"),
                                                    "Field Accuracy": ("exact", "mean")})
    docs = (df.groupby(["model", "dossier", "doc_id", "level", *[b for b in by if b != "level"]], observed=True)
              .exact.all().groupby(keys, observed=True).mean().rename("Document Accuracy"))
    return fields.join(docs)

def calibration(df, bins=(0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.98, 1.0001)):
    df = df.dropna(subset=["score"])
    if df.empty:
        return None, np.nan
    g = df.groupby(pd.cut(df.score, list(bins), right=False), observed=True).agg(
        conf=("score", "mean"), acc=("exact", "mean"), n=("exact", "size"))
    return g, float((g.n / g.n.sum() * (g.acc - g.conf).abs()).sum())

main_e2e, main_rec = main_metrics(e2e), main_metrics(rec)
cer_lv = rec.groupby(["model", "level"], observed=True).cer.mean().unstack("level").reindex(columns=LEVELS)
coef = main_e2e.add_suffix(" (trang)").join(main_rec.add_suffix(" (rec-only)")).assign(**{
    "CER bỏ dấu (rec-only)": rec.groupby("model").cer_nomark.mean(),
    "Độ suy giảm (rec-only)": cer_lv["heavy"] - cer_lv["clean"],
    "ECE (rec-only)": pd.Series({m: calibration(rec[rec.model == m])[1] for m in MODELS}),
    "Trang bị cắt": pages.groupby("model").truncated.mean(),
    "Đọc thừa (rec-only)": rec.groupby("model").over.mean(),
    "Giây/trang": pages.groupby("model").seconds.mean(),
    "Giây/trường": rec.groupby("model").seconds.mean(),
}).loc[MODELS]
higher = [c for c in coef.columns if "Accuracy" in c]
timing = ["Giây/trang", "Giây/trường"]
(coef.style
    .format({c: "{:.1%}" for c in coef.columns if c not in timing} | {c: "{:.1f}" for c in timing}, na_rep="–")
    .highlight_min(subset=[c for c in coef.columns if c not in higher], props="font-weight: bold")
    .highlight_max(subset=higher, props="font-weight: bold"))

### 6.2. Bốn chỉ số chính

In [ ]:
def scorecard(table, title):
    fig, axes = plt.subplots(1, len(MAIN), figsize=(3.2 * len(MAIN) + 2.5, 0.42 * len(MODELS) + 1.4), sharey=True)
    y = np.arange(len(MODELS))
    for i, (ax, (col, hb)) in enumerate(zip(axes, MAIN)):
        vals = table[col].reindex(MODELS).values
        ax.barh(y, vals, height=0.62, color=[COLOR[m] for m in MODELS], edgecolor=SURFACE, linewidth=2)
        for yi, v in zip(y, vals):
            ax.text(v, yi, f" {v:.1%}", va="center", fontsize=9, color=INK)
        ax.set_xlim(0, 1.25 if hb else max(np.nanmax(vals) * 1.45, 0.05))
        ax.set_title(f"{col}\n({'cao' if hb else 'thấp'} là tốt)", loc="left", fontsize=10)
        style_axes(ax, grid="x", percent_axis="x")
        ax.tick_params(labelleft=(i == 0))
    axes[0].set_yticks(y, [LABEL[m] for m in MODELS])
    axes[0].invert_yaxis()
    fig.suptitle(title, x=0.01, ha="left")
    plt.tight_layout()

scorecard(main_e2e, "Cả trang (tìm đoạn khớp nhất trong cả trang, dễ hơn cách khớp của ocr-eval)")
scorecard(main_rec, "Recognition-only (vùng cắt theo bbox ground truth)")

### 6.3. Theo mức nhiễu

Đường dốc mạnh là model nhạy với chất lượng ảnh.

In [ ]:
panels = [("Cả trang", e2e, "CER", False), ("Cả trang", e2e, "Field Accuracy", True),
          ("Rec-only", rec, "CER", False), ("Rec-only", rec, "Field Accuracy", True)]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, (mode, df, col, hb) in zip(axes, panels):
    t = main_metrics(df, by=["level"])
    for m in MODELS:
        s = t.loc[m, col].reindex(LEVELS)
        ax.plot(LEVELS, s.values, color=COLOR[m], linewidth=2, marker=MARKER[m], markersize=7,
                markeredgecolor="white", markeredgewidth=2)
    ax.set_ylim(0, 1.05 if hb else max(t[col].max() * 1.2, 0.05))
    ax.set_title(f"{mode}: {col} ({'cao' if hb else 'thấp'} là tốt)", loc="left")
    style_axes(ax)
plt.tight_layout(rect=(0, 0.12, 1, 1))
legend(fig)

### 6.4. Nguồn lỗi: mặt chữ hay dấu

**Lỗi mặt chữ** = CER bỏ dấu (sai cả khi bỏ dấu). **Lỗi dấu** = CER − CER bỏ dấu. Model đa ngôn ngữ không train nhiều tiếng Việt thường đọc đúng mặt chữ nhưng rơi dấu.

In [ ]:
rows = []
for mode, df in [("cả trang", e2e), ("rec-only", rec)]:
    g = df.groupby("model")[["cer", "cer_nomark"]].mean()
    rows += [dict(model=m, mode=mode, face=g.loc[m, "cer_nomark"], mark=max(g.loc[m, "cer"] - g.loc[m, "cer_nomark"], 0)) for m in MODELS]
err = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 0.42 * len(MODELS) + 1.4), sharey=True)
y = np.arange(len(MODELS))
for ax, mode in zip(axes, ["cả trang", "rec-only"]):
    d = err[err["mode"] == mode].set_index("model").loc[MODELS]
    ax.barh(y, d.face, height=0.62, color="#184f95", edgecolor=SURFACE, linewidth=2, label="Lỗi mặt chữ")
    ax.barh(y, d.mark, left=d.face, height=0.62, color="#86b6ef", edgecolor=SURFACE, linewidth=2, label="Lỗi dấu")
    for yi, t in zip(y, d.face + d.mark):
        ax.text(t, yi, f" {t:.1%}", va="center", fontsize=9)
    ax.set_xlim(0, max((err.face + err.mark).max() * 1.25, 0.05))
    ax.set_title(f"CER {mode}", loc="left")
    style_axes(ax, grid="x", percent_axis="x")
axes[0].set_yticks(y, [LABEL[m] for m in MODELS])
axes[0].invert_yaxis()
axes[1].legend(frameon=False, ncol=2, loc="lower right", bbox_to_anchor=(1, 1))
plt.tight_layout()

### 6.5. Theo kiểu điền và theo trường

CER rec-only theo kiểu điền (viết tay / đánh máy / in sẵn) × mức nhiễu, và `TOP_FIELDS` trường khó nhất, kèm model đọc tốt nhất.

In [ ]:
show = {m: LABEL[m] for m in MODELS}
(rec.pivot_table(index=["fill", "level"], columns="model", values="cer", aggfunc="mean", observed=True)[MODELS]
    .rename(columns=show).style.format("{:.1%}").background_gradient(cmap=SEQ, vmin=0, vmax=0.5))

In [ ]:
TOP_FIELDS = 20
by_field = rec.pivot_table(index=["doc_type", "field", "fill"], columns="model", values="cer", aggfunc="mean", observed=True)[MODELS]
by_field = by_field.loc[by_field.mean(axis=1).sort_values(ascending=False).index[:TOP_FIELDS]].rename(columns=show)
(by_field.assign(**{"tốt nhất": by_field.idxmin(axis=1)})
    .style.format("{:.1%}", subset=list(show.values())).background_gradient(cmap=SEQ, vmin=0, vmax=0.5, subset=list(show.values())))

### 6.6. Hiệu chỉnh điểm tự tin

Rec-only. Model hiệu chỉnh tốt nằm sát đường chéo. Nằm dưới là tự tin quá mức. VLM sinh greedy thường có xác suất token rất cao kể cả khi sai, nên hay tự tin quá mức hơn OCR (xem AIP-002).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot([0, 1], [0, 1], color=MUTED, linewidth=1, linestyle="--")
for m in MODELS:
    g, ece = calibration(rec[rec.model == m])
    if g is None:
        continue
    ax.plot(g.conf, g.acc, color=COLOR[m], linewidth=2, marker=MARKER[m], markersize=7,
            markeredgecolor="white", markeredgewidth=2, label=f"{LABEL[m]} (ECE {ece:.1%})")
ax.set(xlim=(0, 1.02), ylim=(0, 1.02), aspect="equal", xlabel="Điểm tự tin trung bình", ylabel="Field Accuracy thực tế")
ax.set_title("Reliability diagram, recognition-only", loc="left")
style_axes(ax, grid="both")
ax.xaxis.set_major_formatter(pct)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0, -0.14), fontsize=8)
plt.tight_layout()

### 6.7. So với OCR chuyên dụng (rec-only)

Đọc `outputs/scores_rec.csv` do `ocr-eval.ipynb` ghi, chỉ so trên các trường cả hai notebook cùng chấm. Chạy `ocr-eval.ipynb` với cùng `MAX_IMAGES` để phủ nhau nhiều nhất.

Bảng thứ hai trả lời câu hỏi của cascade (AIP-035): trong các trường OCR tốt nhất đọc **sai**, VLM đọc lại **đúng** bao nhiêu, và trong các trường OCR đọc **đúng**, VLM làm hỏng bao nhiêu.

In [ ]:
OCR_SCORES = OUTPUT_DIR / "scores_rec.csv"
KEY = ["image", "field", "occurrence"]
if not OCR_SCORES.exists():
    print(f"Chưa có {OCR_SCORES}: chạy ocr-eval.ipynb trước. Bỏ qua 6.7.")
    ocr = None
else:
    ocr = pd.read_csv(OCR_SCORES, keep_default_na=False)
    ocr["exact"] = ocr.cer == 0
    both = rec[KEY].drop_duplicates().merge(ocr[KEY].drop_duplicates())
    if both.empty:
        print("Không có trường chung giữa hai notebook (khác MAX_IMAGES hoặc khác dữ liệu). Bỏ qua 6.7.")
        ocr = None
if ocr is not None:
    ocr_models = list(dict.fromkeys(ocr.model))
    allrec = pd.concat([ocr.merge(both), rec.merge(both)], ignore_index=True)
    cmp = allrec.groupby("model").agg(CER=("cer", "mean"), **{"Field Accuracy": ("exact", "mean")}).loc[ocr_models + MODELS]
    print(f"{len(both)} trường chung")

    fig, axes = plt.subplots(1, 2, figsize=(12, 0.36 * len(cmp) + 1.4), sharey=True)
    y = np.arange(len(cmp))
    colors = [COLOR.get(m, BASELINE) for m in cmp.index]
    for ax, (col, hb) in zip(axes, [("CER", False), ("Field Accuracy", True)]):
        ax.barh(y, cmp[col], height=0.62, color=colors, edgecolor=SURFACE, linewidth=2)
        for yi, v in zip(y, cmp[col]):
            ax.text(v, yi, f" {v:.1%}", va="center", fontsize=9)
        ax.set_xlim(0, 1.25 if hb else max(cmp[col].max() * 1.45, 0.05))
        ax.set_title(f"{col} rec-only ({'cao' if hb else 'thấp'} là tốt)", loc="left")
        style_axes(ax, grid="x", percent_axis="x")
    axes[0].set_yticks(y, [LABEL.get(m, m) + ("" if m in COLOR else "  (OCR)") for m in cmp.index])
    axes[0].invert_yaxis()
    plt.tight_layout()

In [ ]:
if ocr is not None:
    best_ocr = cmp.loc[ocr_models, "CER"].idxmin()
    base = ocr[ocr.model == best_ocr].merge(both)[KEY + ["exact"]].rename(columns={"exact": "ocr_exact"})
    rows = []
    for m in MODELS:
        d = rec[rec.model == m].merge(base, on=KEY)
        rows.append({"model": LABEL[m],
                     "Cứu được (OCR sai → VLM đúng)": (d.exact & ~d.ocr_exact).sum() / max((~d.ocr_exact).sum(), 1),
                     "Làm hỏng (OCR đúng → VLM sai)": (~d.exact & d.ocr_exact).sum() / max(d.ocr_exact.sum(), 1),
                     "Field Accuracy nếu VLM đọc lại mọi trường OCR sai": (d.exact | d.ocr_exact).mean()})
    print(f"OCR tốt nhất: {best_ocr}, Field Accuracy {base.ocr_exact.mean():.1%} trên {len(base)} trường")
    display(pd.DataFrame(rows).set_index("model").style.format("{:.1%}"))

## 7. Soi lỗi

### 7.1. Kết quả thô của một trang

Chữ model trả về (chưa bỏ định dạng) cho một trang, để thấy model có tự thêm định dạng, bỏ sót vùng, lặp hay dịch không. Đổi `PAGE_INDEX` để xem trang khác.

In [ ]:
PAGE_INDEX, MAX_CHARS = 0, 1500
im = images[PAGE_INDEX]
print(im)
for m in MODELS:
    p = pages[(pages.model == m) & (pages.image == im)].iloc[0]
    print(f"\n===== {LABEL[m]}  ({p.seconds:.0f}s, {p.tokens} token{', BỊ CẮT' if p.truncated else ''})")
    print(p.text[:MAX_CHARS] + (" …" if len(p.text) > MAX_CHARS else ""))

### 7.2. Ký tự hay sai

Rec-only, ảnh sạch. Mỗi model 6 ký tự có tỷ lệ sai cao nhất (xuất hiện ≥ 5 lần), dạng `ký tự tỷ lệ (→ bị đọc thành)`.

In [ ]:
def char_errors(df, top=6, min_seen=5):
    seen, wrong, became = Counter(), Counter(), {}
    for g, h in zip(df.truth.map(norm), df.hyp):
        seen.update(g)
        for op in Levenshtein.editops(g, h):
            if op.tag in ("delete", "replace"):
                c = g[op.src_pos]
                wrong[c] += 1
                became.setdefault(c, Counter())[h[op.dest_pos] if op.tag == "replace" else "∅"] += 1
    worst = sorted((c for c in wrong if c != " " and seen[c] >= min_seen), key=lambda c: -wrong[c] / seen[c])[:top]
    return [f"{c} {wrong[c] / seen[c]:.0%} (→{became[c].most_common(1)[0][0]})" for c in worst]

pd.DataFrame({LABEL[m]: pd.Series(char_errors(rec[(rec.model == m) & (rec.level == "clean")]), dtype=object) for m in MODELS}).T.fillna("")

### 7.3. Cùng một trường, mỗi model đọc ra gì

Rec-only, ảnh sạch. Các trường mà các model chênh CER nhiều nhất.

In [ ]:
wide = rec[rec.level == "clean"].pivot_table(index=["image", "field", "occurrence", "truth"], columns="model",
                                             values=["hyp", "cer"], aggfunc="first")
spread = (wide["cer"].max(axis=1) - wide["cer"].min(axis=1)).sort_values(ascending=False)
wide.loc[spread.index[:15], "hyp"][MODELS].rename(columns=show).reset_index(["image", "occurrence"], drop=True)

### 7.4. Ca sai nặng nhất của một model

Rec-only. Đổi `WORST_MODEL`, `WORST_LEVEL` để xem model / mức nhiễu khác. CER 100% kèm kết quả dài thường là model bịa thêm hoặc trả lời thay vì đọc.

In [ ]:
WORST_MODEL, WORST_LEVEL, N_WORST = MODELS[0], "clean", 10

s = rec[(rec.model == WORST_MODEL) & (rec.level == WORST_LEVEL) & (rec.cer > 0)].sort_values("cer", ascending=False).head(N_WORST)
if s.empty:
    print("Không có ca sai")
else:
    fig, axes = plt.subplots(len(s), 1, figsize=(10, 1.1 * len(s)))
    for ax, r in zip(np.atleast_1d(axes), s.itertuples()):
        ax.imshow(cv2.cvtColor(crop_box(cv2.imread(r.image), r.x, r.y, r.w, r.h), cv2.COLOR_BGR2RGB))
        ax.axis("off")
        ax.set_title(f"[{r.level}/{r.fill}] {r.field}  CER {r.cer:.0%}\nGT : {r.truth}\nVLM: {r.hyp[:120]}",
                     loc="left", fontsize=9, family="monospace")
    plt.tight_layout()

## 8. Ghi chú đọc kết quả

- **Cả trang tốt mà rec-only kém:** vùng cắt nhỏ, thiếu ngữ cảnh, model hay "trả lời" thay vì đọc (ví dụ thêm `Chữ trong ảnh là: ...`). Thử sửa `LINE_PROMPT`, tăng `LINE_MIN_SIDE`, hoặc cắt rộng hơn (`PAD`).
- **Trang bị cắt cao:** model lặp một dòng tới hết `PAGE_MAX_TOKENS`. Hay gặp ở model nhỏ và ảnh nhiễu nặng. Có thể thêm `repetition_penalty` nhẹ (1.05–1.1), đổi lại có thể làm sai các trường có ký tự lặp thật (số `000`, dấu chấm điền chỗ trống).
- **Lỗi dấu lớn:** model đa ngôn ngữ ít dữ liệu tiếng Việt. Vintern được fine-tune cho tiếng Việt nên thường ít lỗi dấu hơn Qwen/Gemma cùng cỡ.
- **Đọc thừa:** kết quả dài hơn ground truth nhiều. Với biểu mẫu là dấu hiệu model chép cả nhãn in sẵn hoặc chữ dòng bên cạnh, hoặc bịa thêm. Đây là rủi ro chính khi dùng VLM cho hồ sơ pháp lý: chữ bịa ra nhìn vẫn hợp lý.
- **Score của VLM** là xác suất token khi sinh greedy, gần 1 kể cả khi sai. Trước khi dùng làm ngưỡng cascade phải hiệu chỉnh (AIP-002).
- **Tốc độ:** trên CPU chỉ để thử đúng/sai. Muốn so tốc độ thật, chạy trên GPU, tốt nhất qua vLLM (engine `vllm`) vì có batching liên tục.
- Chữ viết tay trong golden set là font viết tay, dễ hơn chữ viết tay thật. Cần kiểm lại trên `real/` trước khi chốt model.